# 🛣️ RoadWatch — YOLOv8 Road Damage Detection Training

**Dataset:** RDD2022 (Road Damage Detection 2022)  
**Model:** YOLOv8n (nano) — optimised for edge/server inference  
**Classes:** `pothole`, `crack`, `surface_damage`, `multiple`  

This notebook downloads the public RDD2022 dataset, builds a YOLO-compatible
`data.yaml`, trains for 50 epochs with augmentation, evaluates mAP, and
copies the best checkpoint to Google Drive.

> **Runtime:** Use *GPU* → T4 or better via *Runtime → Change runtime type*.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 1 — Setup: install packages & mount Google Drive
# ═══════════════════════════════════════════════════════════════════
!pip install ultralytics roboflow gdown albumentations -q

import os, sys, shutil, glob, random, yaml
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Ensure output dir exists on Drive
DRIVE_OUT = '/content/drive/MyDrive/roadwatch'
os.makedirs(DRIVE_OUT, exist_ok=True)

print('✅ Setup complete')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 2 — Download & extract the RDD2022 dataset
# ═══════════════════════════════════════════════════════════════════
import gdown, zipfile

DATASET_URL = 'https://drive.google.com/uc?id=1nkDFbLRx1BHnSnCBNeGVFwSJv4xJEMcP'
ZIP_PATH    = '/content/rdd2022.zip'
EXTRACT_DIR = '/content/rdd2022'

if not os.path.exists(EXTRACT_DIR):
    print('⬇️  Downloading RDD2022 dataset …')
    gdown.download(DATASET_URL, ZIP_PATH, quiet=False)

    print('📦 Extracting …')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    os.remove(ZIP_PATH)
    print('✅ Extraction complete')
else:
    print('ℹ️  Dataset already extracted')

# ── Print folder structure (2 levels deep) ──
print('\n📁 Folder structure:')
for root, dirs, files in os.walk(EXTRACT_DIR):
    depth = root.replace(EXTRACT_DIR, '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')
        if len(files) > 5:
            print(f'{indent}  ... and {len(files)-5} more files')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 3 — Auto-detect splits & build data.yaml
# ═══════════════════════════════════════════════════════════════════

def find_image_dir(base, split_name):
    """Walk the tree looking for <split>/images or <split>/."""
    for root, dirs, files in os.walk(base):
        basename = os.path.basename(root).lower()
        parent   = os.path.basename(os.path.dirname(root)).lower()
        # Match: .../train/images  or  .../images  inside a train folder
        if basename == 'images' and split_name in parent:
            return root
        # Match: .../train  that directly contains image files
        if basename == split_name:
            imgs = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]
            if imgs:
                return root
            # Check for images/ subfolder
            if 'images' in dirs:
                return os.path.join(root, 'images')
    return None

train_dir = find_image_dir(EXTRACT_DIR, 'train')
val_dir   = find_image_dir(EXTRACT_DIR, 'val') or find_image_dir(EXTRACT_DIR, 'valid')
test_dir  = find_image_dir(EXTRACT_DIR, 'test')

assert train_dir, f'❌ Could not find train/images under {EXTRACT_DIR}'
assert val_dir,   f'❌ Could not find val/images under {EXTRACT_DIR}'

print(f'Train images : {train_dir}  ({len(os.listdir(train_dir))} files)')
print(f'Val images   : {val_dir}    ({len(os.listdir(val_dir))} files)')
if test_dir:
    print(f'Test images  : {test_dir}   ({len(os.listdir(test_dir))} files)')
else:
    print('Test images  : (none found — will skip test split)')

# Build the yaml relative to the dataset root that contains the splits
# We need 'path' to be the common ancestor of all split dirs
dataset_root = os.path.commonpath([train_dir, val_dir])
# Go up one level if we're inside images/
if os.path.basename(dataset_root) == 'images':
    dataset_root = os.path.dirname(dataset_root)
# Find another common ancestor
dataset_root = os.path.commonpath(
    [os.path.dirname(train_dir), os.path.dirname(val_dir)]
)

data_cfg = {
    'path': dataset_root,
    'train': os.path.relpath(train_dir, dataset_root),
    'val':   os.path.relpath(val_dir,   dataset_root),
    'nc':    4,
    'names': ['pothole', 'crack', 'surface_damage', 'multiple'],
}
if test_dir:
    data_cfg['test'] = os.path.relpath(test_dir, dataset_root)

YAML_PATH = os.path.join(EXTRACT_DIR, 'data.yaml')
with open(YAML_PATH, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f'\n📄 data.yaml written to {YAML_PATH}')
print('--- contents ---')
print(open(YAML_PATH).read())

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 4 — Train YOLOv8n on RDD2022
# ═══════════════════════════════════════════════════════════════════
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # pretrained nano backbone

results = model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    # ── Augmentation ──
    mosaic=1.0,
    flipud=0.3,
    fliplr=0.5,
    hsv_h=0.015,
    # ── Output ──
    project='/content/runs',
    name='roadwatch',
    exist_ok=True,
)

print('\n✅ Training complete')
print(f'Best weights: /content/runs/roadwatch/weights/best.pt')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 5 — Evaluate & Export to Google Drive
# ═══════════════════════════════════════════════════════════════════
from ultralytics import YOLO
import shutil

BEST_PT = '/content/runs/roadwatch/weights/best.pt'

# ── Evaluation ──
best_model = YOLO(BEST_PT)
metrics = best_model.val(data=YAML_PATH, verbose=False)

print('╔══════════════════════════════════════╗')
print('║   RoadWatch — Final Evaluation       ║')
print('╠══════════════════════════════════════╣')
print(f'║  mAP@50     : {metrics.box.map50:.4f}              ║')
print(f'║  mAP@50-95  : {metrics.box.map:.4f}              ║')
print('╚══════════════════════════════════════╝')

# Per-class breakdown
class_names = ['pothole', 'crack', 'surface_damage', 'multiple']
if hasattr(metrics.box, 'maps') and len(metrics.box.maps) >= len(class_names):
    print('\nPer-class mAP@50:')
    for i, name in enumerate(class_names):
        print(f'  {name:20s} → {metrics.box.maps[i]:.4f}')

# ── Export to Google Drive ──
dst = os.path.join(DRIVE_OUT, 'best.pt')
shutil.copy(BEST_PT, dst)
print(f'\n💾 Saved to Google Drive: {dst}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 6 — Validate inference on 3 sample images
# ═══════════════════════════════════════════════════════════════════
import glob, random
from ultralytics import YOLO

best_model = YOLO(BEST_PT)

# Prefer val images; fall back to train
sample_pool = glob.glob(os.path.join(val_dir, '*.jpg')) + \
              glob.glob(os.path.join(val_dir, '*.png'))
if len(sample_pool) < 3:
    sample_pool += glob.glob(os.path.join(train_dir, '*.jpg'))

random.seed(42)
samples = random.sample(sample_pool, min(3, len(sample_pool)))

print('🔍 Inference validation on 3 sample images:\n')
for idx, img_path in enumerate(samples, 1):
    preds = best_model(img_path, conf=0.25, verbose=False)
    r = preds[0]
    fname = os.path.basename(img_path)
    n_boxes = len(r.boxes) if r.boxes is not None else 0

    print(f'━━━ Sample {idx}: {fname} ━━━')
    print(f'    Bounding boxes detected: {n_boxes}')

    if n_boxes == 0:
        print('    (no detections above 0.25 confidence)')
    else:
        for i in range(n_boxes):
            cls_id = int(r.boxes.cls[i])
            conf   = float(r.boxes.conf[i])
            name   = best_model.names.get(cls_id, f'class_{cls_id}')
            print(f'    → {name:20s}  conf={conf:.4f}')
    print()

print('✅ Inference validation complete')